# Use of Force Data Cleaning

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/use_of_force.csv")

In [2]:
df.dtypes

Incident Number              int64
Subject_Incident Number    float64
Sex                            str
Race                           str
Subject Role                   str
Subject Age                float64
Block Address                  str
City                           str
Use of Force Number          int64
Officer_Incident Number      int64
Officer Role                   str
Use of Force Level             str
Use of Force Type              str
OccurredDate                   str
Subject Arrested               str
Use of Force Reason            str
Subject Altered State          str
NonBerkeley Address           bool
dtype: object

## Standardize Column Names & ID Types

In [3]:
# rename columns
rename_map = {"Incident Number": "incident_number", "Subject_Incident Number": "subject_incident_number", 
              "Sex": "sex", "Race": "race", "Subject Role": "subject_role", "Subject Age": "subject_age",
              "Block Address": "block_address", "City": "city", "Use of Force Number": "use_of_force_number",
              "Officer_Incident Number": "officer_incident_number", "Officer Role": "officer_role", 
              "Use of Force Level": "use_of_force_level", "Use of Force Type": "use_of_force_type",
              "OccurredDate": "occurred_date", "Subject Arrested": "subject_arrested", 
              "Use of Force Reason": "use_of_force_reason", "Subject Altered State": "subject_altered_state",
              "NonBerkeley Address": "non_berkeley_address"}
df = df.rename(columns=rename_map)

# convert ID columns to nullable integers 
id_cols = ["incident_number", "subject_incident_number", "use_of_force_number", "officer_incident_number"]
for col in id_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

df.head()

,incident_number,subject_incident_number,sex,race,subject_role,subject_age,block_address,city,use_of_force_number,officer_incident_number,officer_role,use_of_force_level,use_of_force_type,occurred_date,subject_arrested,use_of_force_reason,subject_altered_state,non_berkeley_address
0,4475,3073,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
1,4475,3073,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
2,4475,3072,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
3,4475,3072,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
4,4477,3074,Female,White,Suspect,18.0,2600 Block of Piedmont,Berkeley,8648,10526,Primary Officer,Level 1,Level 1 Grab,2026-11-21,No,Fleeing,Alcohol,False


## Filter to Berkeley & Timeframe (2021-03-01 to 2026-04-30)

In [6]:
# filter city to Berkeley
df["city"] = df["city"].astype(str).str.strip().str.title()
df = df[df["city"] == "Berkeley"].copy()

# filter timeframe
dt_temp = pd.to_datetime(df["occurred_date"], utc=True, errors="coerce")
start_ts = pd.Timestamp("2021-03-01", tz="UTC")
end_ts = pd.Timestamp("2026-04-30 23:59:59", tz="UTC")

df = df[(dt_temp >= start_ts) & (dt_temp <= end_ts)].copy()
dt_temp = dt_temp[(dt_temp >= start_ts) & (dt_temp <= end_ts)]

# format occurred_date to match team spec & extract temporal helper columns
df["occurred_date"] = dt_temp.dt.strftime("%Y-%m-%d %H:%M:%S+00")

# --- verification check ---
print(f"Filtered row count: {len(df):,}")
print(f"Unique City: {df["city"].unique().tolist()}")
print(f"Date range: {df['occurred_date'].min()} to {df['occurred_date'].max()}")

Filtered row count: 7,530
Unique City: ['Berkeley']
Date range: 2021-03-01 00:00:00+00 to 2026-04-30 00:00:00+00


## Categorical & Text Cleanup

In [13]:
na_tokens = ["Unknown", "Not Appl", "Not Applicable", "nan", "None", ""]

# clean Use of Force Level & create numeric version (1-4)
df["use_of_force_level"] = (df["use_of_force_level"].astype(str).str.strip().replace(na_tokens, pd.NA))

level_map = {"Level 1": 1, "Level 2": 2, "Level 3": 3, "Level 4": 4}
df["force_level_num"] = df["use_of_force_level"].map(level_map).astype("Int64")

# clean Use of Force Type typos/spacing & extract force technique excluding the level
df["use_of_force_type"] = (df["use_of_force_type"].astype(str).str.strip()
    .str.replace("Levle 3", "Level 3", regex=False)
    .str.replace("Bodyweight", "Body Weight", regex=False)
    .replace(na_tokens, pd.NA))

df["force_technique"] = df["use_of_force_type"].str.replace(r"^Level \d+\s+", "", regex=True)

# clean Use of Force Reason casing & single NaN
df["use_of_force_reason"] = (df["use_of_force_reason"].astype(str).str.strip()
    .replace({"brandishing a weapon": "Brandishing a Weapon"})
    .replace(na_tokens, pd.NA))

# create boolean Subject Arrested column
df["subject_arrested"] = df["subject_arrested"].astype(str).str.strip().str.title().replace(na_tokens, pd.NA)
df["subject_arrested_bool"] = df["subject_arrested"].map({"Yes": True, "No": False})

# standardize unknowns to NaNs
cat_fill_cols = ["race", "sex", "subject_role", "officer_role", "subject_altered_state"]
for col in cat_fill_cols:
    df[col] = df[col].astype("string").str.strip().replace(na_tokens, pd.NA)


# helper column for counting unique subjects per incident (for cleaning the long format)
df["n_subjects_in_incident"] = df.groupby("incident_number")["subject_incident_number"].transform("nunique")

# --- verification check ---
print("All Columns (Original & New):\n", df.columns.tolist())
print("\nForce Level counts:\n", df["use_of_force_level"].value_counts(dropna=False))
print("\nAge summary:\n", df["subject_age"].describe())
print("\nMissing values remaining:\n", df.isna().sum()[df.isna().sum() > 0])

All Columns (Original & New):
 ['incident_number', 'subject_incident_number', 'sex', 'race', 'subject_role', 'subject_age', 'block_address', 'city', 'use_of_force_number', 'officer_incident_number', 'officer_role', 'use_of_force_level', 'use_of_force_type', 'occurred_date', 'subject_arrested', 'use_of_force_reason', 'subject_altered_state', 'non_berkeley_address', 'force_level_num', 'force_technique', 'subject_arrested_bool', 'n_subjects_in_incident']

Force Level counts:
 use_of_force_level
Level 1    5022
Level 2    2120
Level 3     378
NaN           8
Level 4       2
Name: count, dtype: int64

Age summary:
 count    7264.000000
mean       39.413684
std        13.693111
min         2.000000
25%        30.000000
50%        37.000000
75%        47.000000
max       118.000000
Name: subject_age, dtype: float64

Missing values remaining:
 subject_incident_number      31
sex                          48
race                         87
subject_role                 46
subject_age             

In [12]:
df.head()

,incident_number,subject_incident_number,sex,race,subject_role,subject_age,block_address,city,use_of_force_number,officer_incident_number,...,use_of_force_type,occurred_date,subject_arrested,use_of_force_reason,subject_altered_state,non_berkeley_address,force_level_num,force_technique,subject_arrested_bool,n_subjects_in_incident
21,1463,891,Male,Black,Arrested,50.0,0 Block of Seawall,Berkeley,1075,2371,...,Level 1 Control Hold PCT,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Control Hold PCT,True,2
22,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1077,2372,...,Level 1 Control Hold PCT,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Control Hold PCT,True,2
23,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1078,2372,...,Level 1 Lowered TGSP,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Lowered TGSP,True,2
24,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1079,2373,...,Level 1 Body Weight,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Body Weight,True,2
25,1463,890,Female,Asian,Arrested,28.0,0 Block of Seawall,Berkeley,1080,2371,...,Level 1 Body Weight,2021-04-26 00:00:00+00,Yes,Clearing a Vehicle,<NA>,False,1,Body Weight,True,2
